# Reference-induced leakage on WESAD (wearable stress detection)

Replication notebook for *Reference-Induced Leakage in Subject-Referenced Classification of Repeated Measurements*.

**What it does**
1. Downloads WESAD (Schmidt et al., ICMI 2018; 15 subjects, chest RespiBAN at 700 Hz) and extracts features from non-overlapping 60-s windows.
2. Defines **units** as the five protocol segments of each subject: baseline, amusement, meditation 1, meditation 2 (reference class, *non-stress*) and stress (target class). Windows are repeats of their unit.
3. Measures pseudo-replication (ICC, design effect) and the effect of validation granularity.
4. Compares three subject-referenced designs with leave-one-subject-out (LOSO) validation: **leave-self-out (LSO)**, **unit-wise (UW)** and **size-matched (SM)**, plus unreferenced features and the common practice of per-subject baseline z-scoring.
5. Runs within-subject permutation nulls (the stress label is moved to a random unit of each subject and every reference is rebuilt).
6. Computes the **reference-leakage index** (Corollary 1) and compares it with the empirical null AUC.

**Runtime on Colab (CPU):** about 60-75 min (download ~10 min, features ~10 min, logistic-regression nulls ~15 min, XGBoost nulls ~30 min). Reduce `N_PERM_XGB` for a quicker run.

**Output:** `wesad_outputs.zip` (JSON with every number, CSV tables, a LaTeX table). Upload this zip back to the chat to integrate the results into the manuscript.

In [ ]:
# ---------------------------------------------------------------- configuration
import os, sys, re, json, time, glob, zipfile, pickle, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

TEST_MODE = os.environ.get("WESAD_TEST", "0") == "1"        # synthetic smoke test (not for results)
DATA_DIR  = os.environ.get("WESAD_DIR", "/content/WESAD_data")
OUT_DIR   = os.environ.get("WESAD_OUT", "/content/wesad_outputs")
ZIP_PATH  = os.environ.get("WESAD_ZIP", "")                 # optional: path to an already downloaded WESAD zip
USE_DRIVE = False                                           # set True to look for the zip in Google Drive
DRIVE_ZIP = "/content/drive/MyDrive/WESAD.zip"

FS, WIN_SEC = 700, 60                                       # chest sampling rate; window length (non-overlapping)
SEED = 20261119
N_PERM_LR, N_PERM_XGB, N_PERM_INDEX = 1000, 100, 200
if TEST_MODE:
    N_PERM_LR, N_PERM_XGB, N_PERM_INDEX = 6, 3, 6
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
print("TEST_MODE =", TEST_MODE, "| permutations:", N_PERM_LR, N_PERM_XGB, N_PERM_INDEX)

In [ ]:
# ---------------------------------------------------------------- libraries
from scipy import stats
from scipy.signal import butter, sosfiltfilt, find_peaks, welch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAVE_XGB = False
print("XGBoost available:", HAVE_XGB)

# same XGBoost settings as the bladder analysis (eis_pipeline.py)
XGB_PARAMS = dict(n_estimators=180, max_depth=3, learning_rate=0.04, min_child_weight=3, subsample=0.85,
                  colsample_bytree=0.85, gamma=0.0, reg_alpha=0.10, reg_lambda=2.00,
                  objective="binary:logistic", eval_metric="logloss", tree_method="hist", n_jobs=-1)

def make_model(kind, seed=0):
    if kind == "lr":
        return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
    if HAVE_XGB:
        return XGBClassifier(random_state=seed, **XGB_PARAMS)
    return HistGradientBoostingClassifier(max_iter=180, max_depth=3, learning_rate=0.04,
                                          min_samples_leaf=5, l2_regularization=2.0, random_state=seed)

In [ ]:
# ---------------------------------------------------------------- download and extract
URLS = [
    "https://uni-siegen.sciebo.de/s/pYjSgfOVs6Ntahr/download",                                  # link used in public WESAD repos
    "https://archive.ics.uci.edu/static/public/465/wesad+wearable+stress+and+affect+detection.zip",  # UCI mirror
]

def find_pkls(root):
    return sorted(p for p in glob.glob(os.path.join(root, "**", "S*.pkl"), recursive=True)
                  if re.fullmatch(r"S\d+\.pkl", os.path.basename(p)))

def extract_pkls(zpath, dest):
    with zipfile.ZipFile(zpath) as z:
        for n in z.namelist():
            base = os.path.basename(n)
            if re.fullmatch(r"S\d+\.pkl", base):
                z.extract(n, dest)
            elif n.lower().endswith(".zip"):
                z.extract(n, dest); inner = os.path.join(dest, n)
                extract_pkls(inner, dest); os.remove(inner)

if not TEST_MODE and not find_pkls(DATA_DIR):
    zp = ZIP_PATH
    if USE_DRIVE:
        from google.colab import drive; drive.mount("/content/drive"); zp = DRIVE_ZIP
    if not zp or not os.path.exists(zp):
        zp = os.path.join(DATA_DIR, "WESAD_download.zip")
        for url in URLS:
            print("trying", url)
            r = subprocess.run(["wget", "-q", "--show-progress", "-O", zp, url])
            if r.returncode == 0 and zipfile.is_zipfile(zp):
                break
        else:
            raise SystemExit("Automatic download failed. Download WESAD.zip from "
                             "https://ubi29.informatik.uni-siegen.de/usi/data_wesad.html, put it in Google Drive "
                             "as MyDrive/WESAD.zip, set USE_DRIVE = True above and re-run.")
    print("extracting subject pickles ..."); extract_pkls(zp, DATA_DIR)
PKLS = find_pkls(DATA_DIR)
print(len(PKLS), "subject files:", [os.path.basename(p) for p in PKLS])
assert PKLS, "no S*.pkl files found"

In [ ]:
# ---------------------------------------------------------------- feature extraction (chest device, 60-s windows)
SOS_ECG  = butter(3, [5, 20], btype="band", fs=FS, output="sos")
SOS_RESP = butter(3, 1.0, btype="low", fs=FS, output="sos")
SOS_EDA  = butter(3, 1.0, btype="low", fs=FS, output="sos")
FEATS = ["EDA_mean", "EDA_std", "EDA_slope", "TEMP_mean", "TEMP_slope", "RESP_rate", "RESP_std",
         "ECG_HR", "ECG_RMSSD", "EMG_logstd", "ACC_std"]
COND = {1: "baseline", 2: "stress", 3: "amusement", 4: "meditation"}

def ecg_features(x):
    y = sosfiltfilt(SOS_ECG, x); d = np.diff(y); k = int(0.12 * FS)
    env = np.convolve(d * d, np.ones(k) / k, mode="same")
    pk, _ = find_peaks(env, distance=int(0.33 * FS), height=0.3 * np.percentile(env, 99))
    rr = np.diff(pk) / FS; rr = rr[(rr > 0.3) & (rr < 2.0)]
    if len(rr) < 10:
        return np.nan, np.nan
    return 60.0 / rr.mean(), 1000.0 * np.sqrt(np.mean(np.diff(rr) ** 2))

def slope_per_min(v, fs):
    t = np.arange(len(v)) / fs / 60.0
    return np.polyfit(t, v, 1)[0]

def window_features(ch, a, b):
    eda = ch["EDA"][a:b]; temp = ch["Temp"][a:b]; resp = ch["Resp"][a:b]
    ecg = ch["ECG"][a:b]; emg = ch["EMG"][a:b]; acc = ch["ACC"][a:b]
    eda10 = eda[::70]; temp10 = temp[::70]
    r = resp[::25]; r = r - r.mean(); fr, pr = welch(r, fs=FS / 25, nperseg=len(r))
    band = (fr >= 0.1) & (fr <= 0.7)
    rate = 60.0 * fr[band][np.argmax(pr[band])] if band.any() else np.nan
    hr, rmssd = ecg_features(ecg)
    mag = np.sqrt((acc ** 2).sum(1))
    return [eda10.mean(), eda10.std(), slope_per_min(eda10, 10), temp10.mean(), slope_per_min(temp10, 10),
            rate, r.std(), hr, rmssd, np.log(emg.std() + 1e-9), mag.std()]

def subject_windows(path):
    with open(path, "rb") as f:
        d = pickle.load(f, encoding="latin1")
    ch = {k: np.asarray(v, dtype=np.float64) for k, v in d["signal"]["chest"].items()}
    for k in ["EDA", "Temp", "Resp", "ECG", "EMG"]:
        ch[k] = ch[k].reshape(-1)
    ch["EDA"] = sosfiltfilt(SOS_EDA, ch["EDA"]); ch["Resp"] = sosfiltfilt(SOS_RESP, ch["Resp"])
    lab = np.asarray(d["label"]).reshape(-1).astype(int)
    cp = np.flatnonzero(np.diff(lab)) + 1
    starts, ends = np.r_[0, cp], np.r_[cp, len(lab)]
    rows, count = [], {}
    w = WIN_SEC * FS
    for s0, e0 in zip(starts, ends):
        c = lab[s0]
        if c not in COND or (e0 - s0) < w:
            continue
        count[c] = count.get(c, 0) + 1
        name = COND[c] + (str(count[c]) if c == 4 else ("" if count[c] == 1 else f"_{count[c]}"))
        for j, a in enumerate(range(s0, e0 - w + 1, w)):
            rows.append([os.path.basename(path)[:-4], name, j] + window_features(ch, a, a + w))
    return rows

feat_csv = os.path.join(OUT_DIR, "wesad_window_features.csv")
if os.path.exists(feat_csv) and not TEST_MODE:
    W = pd.read_csv(feat_csv)
else:
    rows = []
    for p in PKLS:
        t0 = time.time(); r = subject_windows(p); rows += r
        print(os.path.basename(p), len(r), "windows", f"{time.time()-t0:.0f}s")
    W = pd.DataFrame(rows, columns=["subject", "cond", "win"] + FEATS)
    W.to_csv(feat_csv, index=False)
n0 = len(W); W = W.dropna(subset=FEATS).reset_index(drop=True)
print(f"windows: {n0}, after removing windows with missing features: {len(W)}")
print(W.groupby("cond").size())

In [ ]:
# ---------------------------------------------------------------- data structures and referencing designs
W["unit"] = W.subject + "|" + W.cond
X = W[FEATS].values.astype(float)
SUBJ = W.subject.values; UNIT = W.unit.values
Y_TRUE = (W.cond == "stress").astype(int).values
SUBJECTS = np.unique(SUBJ)
SUBJ_IDX = {s: np.flatnonzero(SUBJ == s) for s in SUBJECTS}
UNITS_OF = {s: np.unique(UNIT[i]) for s, i in SUBJ_IDX.items()}
print(len(SUBJECTS), "subjects,", W.unit.nunique(), "units, mean windows per unit:",
      round(W.groupby("unit").size().mean(), 2))

def referenced(y, design, Xs=X, stat="median", absval=True):
    # returns Z (rows), src (window index per row); SM gives n_h rows for each target window
    f = (lambda A: np.median(A, axis=0)) if stat == "median" else (lambda A: A.mean(axis=0))
    Z, src = [], []
    for s, idx in SUBJ_IDX.items():
        ref = idx[y[idx] == 0]; ref_units = np.unique(UNIT[ref])
        excl = {u: f(Xs[ref[UNIT[ref] != u]]) for u in ref_units} if design in ("UW", "SM") else None
        full = f(Xs[ref])
        for i in idx:
            if y[i] == 0:
                c = f(Xs[ref[ref != i]]) if design == "LSO" else excl[UNIT[i]]
                Z.append(Xs[i] - c); src.append(i)
            elif design == "SM":
                for u in ref_units:
                    Z.append(Xs[i] - excl[u]); src.append(i)
            else:
                Z.append(Xs[i] - full); src.append(i)
    Z = np.asarray(Z)
    return (np.abs(Z) if absval else Z), np.asarray(src)

def subject_class_weights(subj, y):
    w = np.zeros(len(y))
    for s in np.unique(subj):
        for c in (0, 1):
            m = (subj == s) & (y == c); k = m.sum()
            if k: w[m] = 1.0 / k
    return w * len(y) / w.sum()

def ba(y, p):
    yh = (p >= 0.5).astype(int)
    return 0.5 * ((yh[y == 1] == 1).mean() + (yh[y == 0] == 0).mean())

def loso(Z, src, y, kind, keep=None):
    # leave-one-subject-out; window prob = mean over versions; unit prob = mean over windows
    ys = y[src]; ss = SUBJ[src]; prob = np.zeros(len(src))
    for s in SUBJECTS:
        te = ss == s; tr = ~te
        m = make_model(kind); w = subject_class_weights(ss[tr], ys[tr])
        if kind == "lr": m.fit(Z[tr], ys[tr], logisticregression__sample_weight=w)
        else: m.fit(Z[tr], ys[tr], sample_weight=w)
        prob[te] = m.predict_proba(Z[te])[:, 1]
    pw = pd.Series(prob).groupby(src).mean()
    win = pd.DataFrame({"p": pw.values, "y": y[pw.index.values], "unit": UNIT[pw.index.values]})
    if keep is not None:
        win = win[keep[pw.index.values]]
    un = win.groupby("unit").agg(p=("p", "mean"), y=("y", "first"))
    return ba(un.y.values, un.p.values), ba(win.y.values, win.p.values)

def permuted_labels():
    # within each subject the stress label moves to a randomly chosen unit; references are rebuilt by the caller
    y = np.zeros(len(W), int)
    for s, idx in SUBJ_IDX.items():
        u = rng.choice(UNITS_OF[s]); y[idx[UNIT[idx] == u]] = 1
    return y

In [ ]:
# ---------------------------------------------------------------- 1. pseudo-replication: ICC and design effect
def icc1(groups, x):
    df = pd.DataFrame({"g": groups, "x": x}); k = df.g.nunique(); N = len(df)
    gm = df.groupby("g").x.agg(["mean", "size"]); grand = df.x.mean()
    msb = (gm["size"] * (gm["mean"] - grand) ** 2).sum() / (k - 1)
    msw = ((df.x - df.g.map(gm["mean"])) ** 2).sum() / (N - k)
    n0 = (N - (gm["size"] ** 2).sum() / N) / (k - 1)
    return (msb - msw) / (msb + (n0 - 1) * msw)

R = W.copy()
for f in FEATS:   # remove subject means and condition effects, keep unit-level clustering
    R[f] = R[f] - R.groupby("subject")[f].transform("mean")
    R[f] = R[f] - R.groupby("cond")[f].transform("mean")
size = W.groupby("unit").size(); mbar = size.mean(); cv = size.std(ddof=0) / mbar
icc = pd.DataFrame({"feature": FEATS, "ICC": [icc1(R.unit.values, R[f].values) for f in FEATS]})
icc["DE"] = 1 + ((1 + cv ** 2) * mbar - 1) * icc.ICC; icc["N_eff"] = len(W) / icc.DE
icc.to_csv(os.path.join(OUT_DIR, "wesad_icc.csv"), index=False)
print(icc.round(3).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------- 2. validation granularity (unreferenced features, window level)
def cv_scores(splitter_fn, kind, reps):
    out = []
    for r in range(reps):
        prob = np.zeros(len(W))
        for tr, te in splitter_fn(r):
            m = make_model(kind, seed=r); w = subject_class_weights(SUBJ[tr], Y_TRUE[tr])
            if kind == "lr": m.fit(X[tr], Y_TRUE[tr], logisticregression__sample_weight=w)
            else: m.fit(X[tr], Y_TRUE[tr], sample_weight=w)
            prob[te] = m.predict_proba(X[te])[:, 1]
        out.append(ba(Y_TRUE, prob))
    return float(np.mean(out))

gran = {}
for kind in ["lr", "xgb"]:
    gran[kind] = {
        "random_windows_5fold": cv_scores(lambda r: StratifiedKFold(5, shuffle=True, random_state=r).split(X, Y_TRUE), kind, 5),
        "grouped_by_unit_5fold": cv_scores(lambda r: StratifiedGroupKFold(5, shuffle=True, random_state=r).split(X, Y_TRUE, UNIT), kind, 5),
        "leave_one_subject_out": cv_scores(lambda r: [(np.flatnonzero(SUBJ != s), np.flatnonzero(SUBJ == s)) for s in SUBJECTS], kind, 1),
    }
print(json.dumps(gran, indent=1))

In [ ]:
# ---------------------------------------------------------------- 3. observed accuracy of the designs (true labels)
def baseline_zscore():
    Z = np.zeros_like(X)
    for s, idx in SUBJ_IDX.items():
        b = idx[W.cond.values[idx] == "baseline"]
        Z[idx] = (X[idx] - X[b].mean(0)) / (X[b].std(0) + 1e-9)
    return Z

observed = {}
for kind in ["lr", "xgb"]:
    t0 = time.time(); o = {}
    for d in ["LSO", "UW", "SM"]:
        Z, src = referenced(Y_TRUE, d); o[d] = loso(Z, src, Y_TRUE, kind)
    o["UNREF"] = loso(X, np.arange(len(W)), Y_TRUE, kind)
    Zb = baseline_zscore(); src = np.arange(len(W))
    o["BASEZ_all_units"] = loso(Zb, src, Y_TRUE, kind)
    o["BASEZ_without_baseline_unit"] = loso(Zb, src, Y_TRUE, kind, keep=(W.cond.values != "baseline"))
    observed[kind] = o; print(kind, {k: tuple(round(x, 3) for x in v) for k, v in o.items()}, f"{time.time()-t0:.0f}s")

In [ ]:
# ---------------------------------------------------------------- 4. permutation nulls (references rebuilt in every permutation)
def null_dist(kind, B):
    res = {d: [] for d in ["LSO", "UW", "SM", "UNREF"]}
    t0 = time.time()
    for b in range(B):
        y = permuted_labels()
        for d in ["LSO", "UW", "SM"]:
            Z, src = referenced(y, d); res[d].append(loso(Z, src, y, kind)[0])
        res["UNREF"].append(loso(X, np.arange(len(W)), y, kind)[0])
        if (b + 1) % max(1, B // 10) == 0:
            print(f"  {kind}: {b+1}/{B} permutations, {time.time()-t0:.0f}s")
    return res

nulls = {"lr": null_dist("lr", N_PERM_LR), "xgb": null_dist("xgb", N_PERM_XGB)}
perm_table = []
for kind in nulls:
    for d, v in nulls[kind].items():
        v = np.asarray(v); obs = observed[kind][d][0]
        perm_table.append(dict(classifier=kind, design=d, observed_unit_BA=obs, null_mean=v.mean(),
                               null_lo=np.percentile(v, 2.5), null_hi=np.percentile(v, 97.5),
                               p=(1 + (v >= obs).sum()) / (1 + len(v)), B=len(v)))
perm_table = pd.DataFrame(perm_table); perm_table.to_csv(os.path.join(OUT_DIR, "wesad_permutation.csv"), index=False)
print(perm_table.round(4).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------- 5. reference-leakage index vs empirical null AUC
Xs = X.copy()
for s, idx in SUBJ_IDX.items():               # standardize within subject
    Xs[idx] = (X[idx] - X[idx].mean(0)) / (X[idx].std(0) + 1e-9)
# variance components within subject (units exchangeable under the permutation null)
sw2 = np.zeros(len(FEATS)); sb2 = np.zeros(len(FEATS))
dfw = pd.DataFrame(Xs, columns=FEATS); dfw["unit"] = UNIT; dfw["subject"] = SUBJ
um = dfw.groupby("unit")[FEATS].transform("mean")
sw2 = ((dfw[FEATS] - um) ** 2).sum().values / (len(dfw) - dfw.unit.nunique())
g = dfw.groupby(["subject", "unit"])[FEATS].mean(); ns = dfw.groupby(["subject", "unit"]).size()
sb_list = []
for s in SUBJECTS:
    gm = g.loc[s]; inv_m = (1.0 / ns.loc[s]).mean()
    sb_list.append(gm.var(ddof=1).values - sw2 * inv_m)
sb2 = np.clip(np.mean(sb_list, axis=0), 1e-6, None)
n_h = np.mean([len(UNITS_OF[s]) - 1 for s in SUBJECTS]); m_bar = dfw.groupby("unit").size().mean()

def V(design):
    n = n_h * m_bar - 1
    vt = sb2 * (1 + 1 / n_h) + sw2 * (1 + 1 / (n_h * m_bar))
    if design == "LSO":
        vr = sb2 * m_bar ** 2 * n_h * (n_h - 1) / n ** 2 + sw2 * (1 + 1 / n)
    else:
        vr = sb2 * (1 + 1 / (n_h - 1)) + sw2 * (1 + 1 / ((n_h - 1) * m_bar))
        if design == "SM": vt = vr
    return vr, vt

valid = np.isfinite(sw2) & np.isfinite(sb2) & (sw2 > 1e-8)          # drop constant features
sw2, sb2 = sw2[valid], sb2[valid]; Xs = Xs[:, valid]
index = {"n_h": float(n_h), "m_bar": float(m_bar), "features_used": [f for f, v in zip(FEATS, valid) if v],
         "ICC_within_subject": (sb2 / (sb2 + sw2)).tolist()}

def eff_dim(D):
    C = np.nan_to_num(np.corrcoef(D.T)); np.fill_diagonal(C, 1.0)
    lam = np.clip(np.linalg.eigvalsh(C), 0, None)
    return float(lam.sum() ** 2 / (lam ** 2).sum())
for d in ["LSO", "UW", "SM"]:
    vr, vt = V(d); rho = float(np.exp(np.mean(np.log(vt / vr))))
    Zr, src = referenced(Y_TRUE, d, Xs=Xs, stat="mean", absval=False)
    deff = eff_dim(Zr[Y_TRUE[src] == 0])
    pred = float(stats.f.cdf(rho, deff, deff))
    aucs = []
    for b in range(N_PERM_INDEX):
        y = permuted_labels(); Z, src = referenced(y, d, Xs=Xs, stat="mean", absval=False)
        sc = (Z ** 2 / vr).sum(1); t, r = sc[y[src] == 1], sc[y[src] == 0]
        aucs.append(stats.mannwhitneyu(t, r).statistic / (len(t) * len(r)))
    index[d] = dict(rho=rho, d_eff=deff, predicted_null_AUC=pred, observed_null_AUC=float(np.mean(aucs)),
                    observed_95=np.percentile(aucs, [2.5, 97.5]).tolist())
    print(d, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in index[d].items()})

In [ ]:
# ---------------------------------------------------------------- 6. save results and a LaTeX table
results = dict(dataset="WESAD chest, 60-s non-overlapping windows", test_mode=TEST_MODE, have_xgb=HAVE_XGB,
               n_subjects=int(len(SUBJECTS)), n_units=int(W.unit.nunique()), n_windows=int(len(W)),
               windows_per_cond=W.groupby("cond").size().to_dict(),
               icc=icc.to_dict(orient="list"), granularity=gran,
               observed={k: {d: list(v) for d, v in o.items()} for k, o in observed.items()},
               permutation=perm_table.to_dict(orient="records"), index=index,
               n_perm=dict(lr=N_PERM_LR, xgb=N_PERM_XGB, index=N_PERM_INDEX), seed=SEED)
json.dump(results, open(os.path.join(OUT_DIR, "wesad_results.json"), "w"), indent=1, default=float)

lab = {"LSO": "Leave-self-out", "UW": "Unit-wise", "SM": "Size-matched", "UNREF": "Unreferenced"}
lines = [r"\begin{tabular}{llcccc}", r"\toprule",
         r"Design & Classifier & Observed BA & Null mean (95\% range) & $p$ & Index \\", r"\midrule"]
for d in ["LSO", "UW", "SM", "UNREF"]:
    for kind in ["lr", "xgb"]:
        r = perm_table[(perm_table.design == d) & (perm_table.classifier == kind)].iloc[0]
        ix = f"{index[d]['predicted_null_AUC']:.3f}" if d in index else "--"
        lines.append(f"{lab[d] if kind=='lr' else ''} & {'Logistic' if kind=='lr' else ('XGBoost' if HAVE_XGB else 'Boosting (HGB)')} & "
                     f"{r.observed_unit_BA:.3f} & {r.null_mean:.3f} ({r.null_lo:.3f}--{r.null_hi:.3f}) & "
                     f"{r.p:.3g} & {ix if kind=='lr' else ''} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(OUT_DIR, "wesad_table.tex"), "w").write("\n".join(lines))
print("\n".join(lines))

subprocess.run(["bash", "-c", f"cd {os.path.dirname(OUT_DIR) or '.'} && zip -qr wesad_outputs.zip {os.path.basename(OUT_DIR)}"])
print("saved:", os.path.join(os.path.dirname(OUT_DIR), "wesad_outputs.zip"))
if not TEST_MODE:
    try:
        from google.colab import files; files.download(os.path.join(os.path.dirname(OUT_DIR), "wesad_outputs.zip"))
    except Exception:
        pass